# Aula 6 — Indicadores, visualização e storytelling

Na Aula 5 a gente montou uma tabela-resumo por hashtag: quantos posts, curtidas médias, engajamento médio, visualizações totais. Uma tabela de número já responde pergunta, mas quem lê rápido (um professor, um cliente, o resto da turma) enxerga um padrão muito mais rápido num gráfico do que numa tabela de 18 linhas.

Hoje a gente pega essa mesma tabela e transforma ela em visualização, com `matplotlib`. A aula segue um fio único: revisão rápida do que veio da Aula 5 (dado que não conseguimos avançar nela), refazer a tabela-resumo (rapidinho, é revisão), a diferença entre indicador e métrica, três tipos de gráfico (barras, linhas, dispersão), boas práticas de visualização e uma palavra sobre dashboards. No fim, cada gráfico feito aqui deve responder a **uma pergunta específica**, sem forçar a leitura além do que os dados mostram.

## 1. Revisão rápida: o que veio da Aula 5

Só pra alinhar antes de seguir:

- **Métrica** é um número calculado a partir dos dados brutos (curtidas médias, taxa de engajamento, quantidade de posts).
- **Filtro** é manter só as linhas que cumprem uma condição (`df[df["plays"] > 0]`).
- **Agrupamento** (`.groupby()`) junta linhas que compartilham um valor (a mesma hashtag) e calcula uma métrica por grupo.
- **Tipo numérico** importa porque só dá pra fazer conta (média, soma, divisão) em coluna que o Pandas reconhece como número (`int64`, `float64`), não em coluna de texto.
- **Indicador** é uma informação construída a partir de uma ou mais métricas para avaliar o desempenho em relação a um objetivo.

Se algum desses termos ainda está vago, essa é a hora de perguntar, porque hoje a gente usa os quatro sem reexplicar.

## 2. Preparando o ambiente

Esta aula usa `pandas` (já conhecido) e `matplotlib`, a biblioteca mais usada pra gráfico em Python. As dependências estão no `requirements.txt` da **raiz** do repositório.

**Não crie um `.venv` novo dentro desta pasta da aula.** Use o ambiente que você já criou na raiz.

No terminal (o integrado do VS Code funciona bem aqui), **dentro da pasta da disciplina** (a pasta clonada, onde já existe o `.venv`):

```cmd
uv pip install -r requirements.txt
```

Se o `uv` não funcionar:

```cmd
pip install -r requirements.txt
```

Os mesmos comandos funcionam no Mac (Terminal).

**ATENÇÃO:** confira no terminal se você está mesmo na pasta raiz (ela costuma terminar com algo como `extracao_analise_2026`). No VS Code/Cursor, o kernel do notebook deve apontar para o `.venv` dessa raiz.


## 3. Recarregando os dados e refazendo a tabela-resumo

Isso é revisão da Aula 5, comprimida em poucas células: carregar, tirar duplicata, filtrar visualização zero, calcular taxa de engajamento, explodir hashtags e agrupar. Se algum passo aqui parecer novo, vale revisitar o notebook da Aula 5.

**OBS:** o arquivo `dados/exportacao.csv` desta pasta é a mesma exportação **real** da Aula 5 (vários perfis do TikTok misturados), pra todo mundo rodar com o mesmo material. No exercício, você troca pela sua coleta.


In [3]:
import pandas as pd  # pandas pra tabela, de novo
import matplotlib.pyplot as plt  # matplotlib pra gráfico, a novidade de hoje

df = pd.read_csv("dados/exportacao.csv", sep=";")  # lê a exportação, mesmo arquivo e mesmo separador da Aula 5
df = df.drop_duplicates()  # remove linha idêntica a outra anterior, se existir

print(f"Linhas depois de remover duplicatas: {len(df)}")

Linhas depois de remover duplicatas: 640


In [4]:
# filtra fora posts sem visualização (senão a divisão da taxa de engajamento quebra) e calcula a taxa
df_posts = df[df["plays"] > 0].copy()  # só posts com denominador válido

df_posts["taxa_engajamento"] = (
    df_posts["likes"] + df_posts["comments"] + df_posts["shares"]
) / df_posts["plays"]  # mesma fórmula da Aula 5: interações dividido por visualizações

print(f"Posts com visualização válida: {len(df_posts)}")

Posts com visualização válida: 640


In [5]:
df_hashtags = df_posts.dropna(subset=["hashtags"]).copy()  # só posts com hashtag preenchida
df_hashtags["hashtags"] = df_hashtags["hashtags"].str.split(",")  # transforma a string em lista
df_hashtags = df_hashtags.explode("hashtags")  # uma linha por hashtag
df_hashtags["hashtags"] = df_hashtags["hashtags"].str.strip().str.lower()  # padroniza minúsculas  # tira espaço sobrando

resumo_hashtags = df_hashtags.groupby("hashtags").agg(  # agrupa por hashtag
    qtd_posts=("id", "count"),  # quantos posts usaram essa hashtag
    curtidas_medias=("likes", "mean"),  # média de curtidas
    engajamento_medio=("taxa_engajamento", "mean"),  # média da taxa de engajamento
    plays_total=("plays", "sum"),  # soma de visualizações
).reset_index().sort_values("engajamento_medio", ascending=False)  # ordena da maior pra menor engajamento

resumo_hashtags.head(10)  # confere as 10 primeiras linhas da tabela-resumo


,hashtags,qtd_posts,curtidas_medias,engajamento_medio,plays_total
510,rock,1,7817.000000,0.251505,31900
357,joiasdobairo,1,7817.000000,0.251505,31900
384,major,1,7817.000000,0.251505,31900
640,ídolo,1,2746.000000,0.248435,11500
23,alextelles,1,2746.000000,0.248435,11500
440,ny,1,3120.000000,0.222778,14400
599,uniforme,1,1198.000000,0.216616,5609
366,lançamento,1,1198.000000,0.216616,5609
453,oídolofica,3,2442.666667,0.201467,41166
271,fyp,2,220285.500000,0.200515,3334600


**O que observar:** essa é a mesma tabela `resumo_hashtags` da Aula 5, só que recalculada aqui do zero. Se os números baterem com o que você viu na aula passada, o pipeline está correto e dá pra seguir pros gráficos.

## 4. Indicador não é a mesma coisa que métrica

As duas palavras aparecem misturadas o tempo todo, mas a diferença importa pra quem visualiza dado:

- **Métrica** é qualquer número calculado a partir dos dados (curtidas médias, taxa de engajamento, quantidade de posts). Existem dezenas de métricas possíveis pra uma coleta.
- **Indicador** é a métrica que você escolheu acompanhar porque ela responde a uma pergunta que importa pra alguma decisão (fizemos uma boa escolha de hashtag? vale a pena investir mais nessa pauta?). Nem toda métrica vira indicador, só as que você decide monitorar de propósito.

Na prática: `curtidas_medias` é uma métrica entre várias que a tabela calcula. Se o objetivo do relatório é "decidir em qual hashtag investir mais conteúdo", `engajamento_medio` é o indicador que a gente escolhe destacar, porque é ele que responde essa pergunta específica. Cada gráfico que vem a seguir foi escolhido pra deixar um indicador claro, não só pra mostrar número por mostrar.

## 5. Gráfico de barras: comparar categorias

Barra é o gráfico certo quando você quer comparar um número entre categorias diferentes (aqui, hashtags). Vamos responder: **quais hashtags têm o maior engajamento médio na coleta?**

In [6]:
top10 = resumo_hashtags.head(10)  # as 10 hashtags com maior engajamento médio (a tabela já veio ordenada)

fig, ax = plt.subplots(figsize=(9, 5))  # cria a figura e os eixos, com um tamanho legível

ax.bar(top10["hashtags"], top10["engajamento_medio"] * 100, color="#3b6ea5")  # barra por hashtag; multiplicamos por 100 pra virar porcentagem

ax.set_title("Engajamento médio por hashtag (top 10)")  # título informativo: diz o que o gráfico mostra
ax.set_xlabel("Hashtag")  # rótulo do eixo horizontal
ax.set_ylabel("Engajamento médio (%)")  # rótulo do eixo vertical, com a unidade
ax.tick_params(axis="x", rotation=45)  # gira os nomes das hashtags pra não sobrepor
fig.text(0.01, -0.02, "Fonte: exportação real TikTok via Zeeschuimer (Aulas 4/5), dados/exportacao.csv", fontsize=8, color="gray")  # fonte dos dados, sempre

fig.tight_layout()  # ajusta os espaçamentos pra nada cortar
plt.show()


<Figure size 900x500 with 1 Axes>

**O que observar:** a hashtag no topo tem o maior engajamento médio da coleta, mas repare que `qtd_posts` (quantos posts usaram ela) não aparece nesse gráfico. Em coleta real, o topo costuma vir cheio de hashtags com 1 post só (um vídeo que performou bem "arrasta" a média). Uma hashtag no topo com só 1 ou 2 posts é um indício mais frágil do que uma no topo com dezenas de posts. **Conclusão que esse gráfico permite:** entre as hashtags analisadas, a do topo teve o maior engajamento médio nesta coleta. **Conclusão que ele NÃO permite:** que usar essa hashtag garante mais engajamento no futuro, ou fora dessa amostra.


## 6. Gráfico de linhas: acompanhar ao longo do tempo

Linha é o gráfico certo quando o eixo horizontal é tempo (data, hora) e você quer ver tendência ou variação. Vamos responder: **o engajamento médio dos posts mudou de um dia pra outro, dentro do período coletado?**

**ATENÇÃO:** esse dataset cobre só alguns dias (~100), e é travado no tempo, ou seja, só foi feito no dia que fiz a raspagem e acabou, então a linha vai ter o mesmo número de pontos que de linhas. Se você usar um dado diferente, o gráfico será diferente, por óbvio. 

In [8]:
df_posts["data_coleta"] = pd.to_datetime(df_posts["timestamp"]).dt.date  # extrai só a data (sem hora) do timestamp completo

engajamento_por_dia = df_posts.groupby("data_coleta")["taxa_engajamento"].mean().reset_index()  # média da taxa de engajamento, por dia
engajamento_por_dia = engajamento_por_dia.sort_values("data_coleta")  # garante a ordem cronológica no eixo

engajamento_por_dia

,data_coleta,taxa_engajamento
0,2024-06-19,0.060854
1,2024-07-08,0.019904
2,2024-07-09,0.092637
3,2024-07-15,0.065618
4,2024-07-17,0.077521
...,...,...
106,2026-08-07,0.083488
107,2026-08-08,0.070597
108,2026-08-09,0.059080
109,2026-08-10,0.053395


In [9]:
fig, ax = plt.subplots(figsize=(9, 5))  # nova figura, um gráfico por vez

ax.plot(
    engajamento_por_dia["data_coleta"],
    engajamento_por_dia["taxa_engajamento"] * 100,
    marker="o",  # marca cada ponto de dado, importante quando são poucos dias
    color="#c0392b",
)

ax.set_title("Engajamento médio por dia de coleta")  # o que o gráfico mostra
ax.set_xlabel("Data")  # rótulo do eixo horizontal
ax.set_ylabel("Engajamento médio (%)")  # rótulo do eixo vertical, com unidade
ax.tick_params(axis="x", rotation=30)  # datas legíveis
fig.text(0.01, -0.02, "Fonte: exportação real TikTok via Zeeschuimer (Aulas 4/5), dados/exportacao.csv", fontsize=8, color="gray")

fig.tight_layout()
plt.show()


<Figure size 900x500 with 1 Axes>

**O que observar:** Existem alguns dados abaixo de 2026, que causam um problema na visualização. Além do mais, existem dias com muito engajamento, e dias com pouco, e a média varia com isso. 

**Conclusão que esse gráfico permite:** descrever como o engajamento médio variou dia a dia dentro do período coletado. **Conclusão que ele NÃO permite:** afirmar uma tendência de longo prazo, ou prever o próximo dia, dado que ele é instável (mas também podemos considerar remover esses dados. Vamos tentar?)

In [12]:
fig, ax = plt.subplots(figsize=(9, 5))  # nova figura, um gráfico por vez

# data_coleta é datetime.date (veio do .dt.date), então o corte também precisa ser data, não texto
engajamento_por_dia = engajamento_por_dia[engajamento_por_dia["data_coleta"] >= pd.to_datetime("2026-04-01").date()]

ax.plot(
    engajamento_por_dia["data_coleta"],
    engajamento_por_dia["taxa_engajamento"] * 100,
    marker="o",  # marca cada ponto de dado, importante quando são poucos dias
    color="#c0392b",
)

ax.set_title("Engajamento médio por dia de coleta")  # o que o gráfico mostra
ax.set_xlabel("Data")  # rótulo do eixo horizontal
ax.set_ylabel("Engajamento médio (%)")  # rótulo do eixo vertical, com unidade
ax.tick_params(axis="x", rotation=30)  # datas legíveis
fig.text(0.01, -0.02, "Fonte: exportação real TikTok via Zeeschuimer (Aulas 4/5), dados/exportacao.csv", fontsize=8, color="gray")

fig.tight_layout()
plt.show()


<Figure size 900x500 with 1 Axes>

## 7. Gráfico de dispersão: relação entre duas variáveis

Dispersão (scatter) é o gráfico certo quando você quer ver se duas variáveis numéricas se relacionam, um ponto por post. Vamos responder: **contas com mais seguidores têm taxa de engajamento maior ou menor?**

In [13]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(
    df_posts["author_followers"],
    df_posts["taxa_engajamento"] * 100,
    alpha=0.6,  # transparência: ajuda a ver onde os pontos se sobrepõem
    color="#27824c",
)

ax.set_title("Seguidores do autor vs. taxa de engajamento do post")  # pergunta que o gráfico responde, no título
ax.set_xlabel("Seguidores do autor")
ax.set_ylabel("Taxa de engajamento (%)")
ax.set_xscale("log")  # escala logarítmica: sem ela, as poucas contas gigantes esmagam as pequenas num canto do gráfico
fig.text(0.01, -0.02, "Fonte: exportação real TikTok via Zeeschuimer (Aulas 4/5), dados/exportacao.csv", fontsize=8, color="gray")

fig.tight_layout()
plt.show()


<Figure size 900x500 with 1 Axes>

**O que observar:** nesta coleta real, as contas gigantes (g1 e CazéTV, na casa dos 12 a 14 milhões de seguidores) aparecem com taxa de engajamento mais baixa, enquanto contas menores ou de nicho (Botafogo, por exemplo) sobem mais no eixo Y. Ou seja, ter mais seguidores **não** veio junto com maior taxa de engajamento aqui, o olho vê o contrário. Repare também no `ax.set_xscale("log")`: sem escala logarítmica, a diferença entre dezenas de seguidores e 14 milhões some num canto do gráfico (as contas grandes esmagam as pequenas). É um exemplo concreto de por que escala é uma decisão de leitura, não só estética.

**Conclusão que esse gráfico permite:** descrever que, nesta coleta, contas maiores tendem a ter taxa de engajamento menor (ou, no mínimo, não maior). **Conclusão que ele NÃO permite:** afirmar causa (ter mais seguidores *causa* menos engajamento), nem generalizar pra outras plataformas ou outras coletas. Correlação não é a mesma coisa que causalidade.


## 8. Boas práticas de visualização

Os três gráficos acima já seguiram essas práticas, mas vale deixar explícito o que checar antes de mostrar um gráfico pra alguém:

- **Título informativo:** diz o que o gráfico mostra, não só o nome da variável (`"Engajamento médio por hashtag"`, não `"Gráfico 1"`).
- **Rótulos nos eixos:** com a unidade, quando fizer sentido (`"Engajamento médio (%)"`, não só `"Engajamento"`).
- **Escala adequada:** decida entre linear e logarítmica pensando em quem vai ler (viu no gráfico de dispersão acima: sem escala log, a diferença entre contas pequenas e grandes desaparece).
- **Cor com propósito:** cor separa categoria ou destaca algo, não é decoração. Poucas cores, e todas legíveis (evite vermelho/verde puro juntos, por exemplo, por causa de daltonismo).
- **Fonte dos dados:** todo gráfico que sai daqui pro Projeto 2 (ou qualquer relatório) leva a origem dos dados escrita nele, igual fizemos com `fig.text(...)` nos três gráficos acima.
- **Uma pergunta, uma conclusão, sem extrapolar:** cada gráfico deveria caber numa frase do tipo "esse gráfico mostra X, entre os dados coletados". Se a frase que você quer escrever generaliza pra fora da amostra ("hashtag X sempre engaja mais"), o gráfico não sustenta essa frase sozinho.

## 9. Dashboards e narrativa orientada a decisão

Um **dashboard** é um painel com vários gráficos e indicadores juntos, atualizados com frequência, feito pra alguém acompanhar uma situação sem precisar reabrir a análise do zero toda vez (ex.: um painel de redes sociais que a equipe de comunicação olha toda semana). Ferramentas comuns pra montar isso incluem Google Data Studio/Looker Studio, Power BI e bibliotecas Python como Streamlit, além do próprio Excel/Planilhas em versões mais simples.

Construir um dashboard de verdade foge do tempo desta aula (é conteúdo mais avançado), mas a ideia central importa desde já: **um gráfico isolado responde uma pergunta, um dashboard organiza várias perguntas em torno de uma decisão**. Antes de montar qualquer painel, vale perguntar: quem vai olhar isso, com que frequência, e que decisão essa pessoa toma a partir do que está vendo? Um dashboard cheio de gráfico bonito mas sem decisão associada é só ruído visual bem organizado.

## 10. Quando der errado

- Gráfico não aparece, ou aparece uma célula de código sem imagem: confira se tem `plt.show()` no final da célula (no Jupyter geralmente aparece sozinho, mas é bom hábito colocar).
- `KeyError` numa coluna: confira o nome exato com `df_posts.columns`, igual na Aula 5. Nome de coluna errado é o erro mais comum.
- Barras ou linha vazia: confira se a tabela usada (`top10`, `engajamento_por_dia`) realmente tem linhas, com um `print(len(...))` antes do gráfico.
- Eixo com hashtag ilegível, tudo espremido: aumente `figsize=(largura, altura)` no `plt.subplots()`, ou aumente a rotação em `tick_params(axis="x", rotation=...)`.
- Gráfico de dispersão com todos os pontos amontoados num canto: é sinal de que a escala linear está escondendo a diferença entre valores pequenos e grandes, tente `ax.set_xscale("log")` ou `ax.set_yscale("log")`.
- `TypeError` ao converter a coluna de data: confira se `pd.to_datetime()` está recebendo a coluna certa, e se o formato da data no seu CSV é parecido com o do exemplo (`2026-08-04 14:05:00`).

## Prática: faça agora

Abra `exercicios/exercicio-06-visualizacao-storytelling.ipynb`. Ele continua o **Projeto 2**: pega a tabela-resumo por hashtag que você já tem (da Aula 5) e acrescenta pelo menos duas visualizações a ela, justificando no README qual pergunta cada gráfico responde.